In [1]:
import os
from pathlib import Path
import uuid
import re

import pandas as pd
import numpy as np
from unidecode import unidecode


**CAREFULL**  
This is the correct notebook that generates the dataset_no0min.csv 

In [2]:
# Paths
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

TRAINING_PATH = OUTPUT_DIR / "training_data_v2.csv"
UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping.csv"
CLEANED_UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping_cleaned.csv"
MASTER_TEAM_LIST_PATH = Path("master_team_list.csv")

print(" DATA_ROOT:", DATA_ROOT.resolve())
print(" OUTPUT_DIR:", OUTPUT_DIR.resolve())
print(" training_data_v2 path:", TRAINING_PATH)


 DATA_ROOT: C:\Users\skourako\diplo\fpl_pipeline\data
 OUTPUT_DIR: C:\Users\skourako\diplo\fpl_pipeline\output
 training_data_v2 path: output\training_data_v2.csv


In [3]:
# %%– Normalization function
def normalize_player_name(name: str) -> str:
    """
    Normalizes player names:
    - to lowercase
    - remove accents
    - strip trailing numbers (e.g. ' 534')
    - remove underscores
    - keep only alphanumeric + spaces
    - collapse multiple spaces
    """
    if pd.isna(name):
        return name

    name = str(name).strip().lower()
    name = unidecode(name)

    # Remove trailing numeric suffixes: "aaron connolly 534", "aaron_connolly_534"
    name = re.sub(r'[\s_]*\d+\s*$', '', name)

    # Replace underscores with spaces
    name = name.replace("_", " ")

    # Keep only alphanumeric + space
    name = "".join(c for c in name if c.isalnum() or c.isspace())

    # Collapse multiple spaces
    name = " ".join(name.split())

    return name


print(" Testing normalize_player_name:")
for t in ["aaron cresswell 376", "Aaron_Connolly534", "Son Heung-Min 123"]:
    print(f"  '{t}' -> '{normalize_player_name(t)}'")


 Testing normalize_player_name:
  'aaron cresswell 376' -> 'aaron cresswell'
  'Aaron_Connolly534' -> 'aaron connolly'
  'Son Heung-Min 123' -> 'son heungmin'


In [4]:
# Updated Loading function for 2020-2026
def load_all_gws(data_root=DATA_ROOT):
    all_seasons = []
    # Explicit list of seasons to include
    target_seasons = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
    
    print(f"Loading GW data for seasons: {target_seasons}")

    for season in target_seasons:
        season_path = data_root / season
        gws_path = season_path / "gws"

        if not season_path.is_dir() or not gws_path.exists():
            print(f"Skipping {season}: Folder or 'gws' subfolder not found.")
            continue

        gw_files = sorted(
            [f for f in os.listdir(gws_path) if f.startswith("gw") and f.endswith(".csv")],
            key=lambda x: int(x.replace("gw","").replace(".csv",""))
        )

        print(f"Season {season}: {len(gw_files)} GWs found.")

        frames = []
        for fname in gw_files:
            gw = int(fname.replace("gw","").replace(".csv",""))
            df = pd.read_csv(gws_path / fname)

            # Standardized columns for consistency across seasons
            keep = [
                "name", "element", "minutes", "goals_scored", "assists", "clean_sheets",
                "goals_conceded","yellow_cards","red_cards","total_points",
                "influence","creativity","threat","ict_index",
                "opponent_team","was_home"
            ]
            cols = [c for c in keep if c in df.columns]
            df = df[cols].copy()

            df["season"] = season
            df["Gameweek"] = gw
            frames.append(df)

        if frames:
            all_seasons.append(pd.concat(frames, ignore_index=True))

    if not all_seasons:
        raise ValueError("No data was loaded. Check your DATA_ROOT path.")
        
    df_gws = pd.concat(all_seasons, ignore_index=True)
    print(f"Total rows loaded: {len(df_gws):,}")
    return df_gws

df_gws = load_all_gws()
display(df_gws.head())
print(df_gws["season"].value_counts().sort_index())


Loading GW data for seasons: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Season 2020-21: 38 GWs found.


Season 2021-22: 38 GWs found.
Season 2022-23: 38 GWs found.


C:\Users\skourako\AppData\Local\Temp\ipykernel_30736\3985452101.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_seasons.append(pd.concat(frames, ignore_index=True))


Season 2023-24: 38 GWs found.
Season 2024-25: 38 GWs found.
Season 2025-26: 34 GWs found.
Total rows loaded: 160,370


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,influence,creativity,threat,ict_index,opponent_team,was_home,season,Gameweek
0,Aaron Connolly,78,45,0,0,0,2,0,0,1,1.2,0.3,32.0,3.4,5,True,2020-21,1
1,Aaron Cresswell,435,90,0,0,0,2,0,0,1,10.4,11.2,0.0,2.2,14,True,2020-21,1
2,Aaron Mooy,60,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,5,True,2020-21,1
3,Aaron Ramsdale,483,90,0,0,0,2,0,0,1,18.2,0.0,0.0,1.8,20,True,2020-21,1
4,Abdoulaye Doucouré,512,90,0,0,1,0,0,0,3,20.4,44.6,4.0,6.9,17,False,2020-21,1


season
2020-21    24365
2021-22    25447
2022-23    26505
2023-24    29725
2024-25    27605
2025-26    26723
Name: count, dtype: int64


In [5]:
#– Φόρτωμα players_raw ανά σεζόν
def load_players_raw_by_season():
    players = {}
    print("\n Loading players_raw per season...")

    for season in sorted(df_gws["season"].unique()):
        path = DATA_ROOT / season / "players_raw.csv"
        if not path.exists():
            print(f"  ⚠️ No players_raw for {season}")
            continue

        df = pd.read_csv(path)

        # unify ID col
        if "id" in df.columns:
            df.rename(columns={"id": "element"}, inplace=True)

        keep = ["element","team","element_type","web_name","first_name","second_name"]
        keep = [c for c in keep if c in df.columns]
        df = df[keep].copy()

        df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
        if "team" in df.columns:
            df["team"] = pd.to_numeric(df["team"], errors="coerce").astype("Int64")

        players[season] = df.set_index("element")
        print(f"  → {season}: {len(df)} players in players_raw")

    return players


players_raw_by_season = load_players_raw_by_season()



 Loading players_raw per season...
  → 2020-21: 713 players in players_raw
  → 2021-22: 737 players in players_raw
  → 2022-23: 778 players in players_raw
  → 2023-24: 865 players in players_raw
  → 2024-25: 804 players in players_raw
  → 2025-26: 830 players in players_raw


In [6]:
# %% Teams per season (με master_team_list fallback για 2018-19)
master_teams_df = pd.read_csv(MASTER_TEAM_LIST_PATH) if MASTER_TEAM_LIST_PATH.exists() else None
if master_teams_df is not None:
    master_teams_df.columns = [c.lower() for c in master_teams_df.columns]

def load_teams_for_season(season: str) -> pd.DataFrame:
    """
    Επιστρέφει DF με στήλες: Team ID, Team Name, short_name
    - teams.csv αν υπάρχει
    - αλλιώς master_team_list.csv (π.χ. 2018-19)
    """
    path = DATA_ROOT / season / "teams.csv"

    if path.exists():
        df = pd.read_csv(path)
        idcol = "id" if "id" in df.columns else "code"
        df.rename(columns={idcol: "Team ID", "name": "Team Name"}, inplace=True)
        df["Team ID"] = pd.to_numeric(df["Team ID"], errors="coerce").astype("Int64")
        if "short_name" not in df.columns:
            df["short_name"] = df["Team Name"]
        return df[["Team ID", "Team Name", "short_name"]]

    if master_teams_df is not None:
        sub = master_teams_df[master_teams_df["season"] == season].copy()
        if not sub.empty:
            sub.rename(columns={"team": "Team ID", "team_name": "Team Name"}, inplace=True)
            sub["Team ID"] = pd.to_numeric(sub["Team ID"], errors="coerce").astype("Int64")
            sub["short_name"] = sub["Team Name"]
            print(f"  🔁 Using master_team_list for {season}")
            return sub[["Team ID", "Team Name", "short_name"]]

    raise RuntimeError(f"❌ No team info for {season}")


In [7]:
#– Φόρτωμα fixtures σε “long” μορφή (home/away rows)
def load_fixtures_long():
    rows = []
    print("\n Building fixture_long from fixtures.csv ...")

    for season in sorted(df_gws["season"].unique()):
        fx_path = DATA_ROOT / season / "fixtures.csv"
        if not fx_path.exists():
            print(f"  ⚠️ No fixtures.csv for {season}, skipping.")
            continue

        fx = pd.read_csv(fx_path)
        if "event" in fx.columns:
            fx["Gameweek"] = fx["event"]
        elif "round" in fx.columns:
            fx["Gameweek"] = fx["round"]
        else:
            raise RuntimeError(f"No event/round column in fixtures for {season}")

        for _, r in fx.iterrows():
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_h"),
                "Opponent ID": r.get("team_a"),
                "Is Home": True,
                "Difficulty": r.get("team_h_difficulty", np.nan)
            })
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_a"),
                "Opponent ID": r.get("team_h"),
                "Is Home": False,
                "Difficulty": r.get("team_a_difficulty", np.nan)
            })

    fixture_long = pd.DataFrame(rows)
    fixture_long["Gameweek"] = pd.to_numeric(fixture_long["Gameweek"], errors="coerce").astype("Int64")
    fixture_long["Team ID"] = pd.to_numeric(fixture_long["Team ID"], errors="coerce").astype("Int64")
    fixture_long["Opponent ID"] = pd.to_numeric(fixture_long["Opponent ID"], errors="coerce").astype("Int64")
    fixture_long["Is Home"] = fixture_long["Is Home"].astype(bool)

    print("fixture_long rows:", len(fixture_long))
    return fixture_long


fixture_long = load_fixtures_long()
display(fixture_long.head())



 Building fixture_long from fixtures.csv ...


fixture_long rows: 4560


,season,Gameweek,Team ID,Opponent ID,Is Home,Difficulty
0,2020-21,1,8,1,True,3
1,2020-21,1,1,8,False,2
2,2020-21,1,6,16,True,2
3,2020-21,1,16,6,False,3
4,2020-21,1,11,10,True,3


In [8]:
# Build df_core per season by deriving Player Team ID from fixtures (perfect fix)
print("Building df_core per season using fixture-based team identification...")

def build_season_core(season: str, df_season: pd.DataFrame) -> pd.DataFrame:
    print(f"Season {season}...")

    df = df_season.copy()
    df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
    df["Gameweek"] = pd.to_numeric(df["Gameweek"], errors="coerce").astype("Int64")
    df["opponent_team"] = pd.to_numeric(df["opponent_team"], errors="coerce").astype("Int64")

    # players_raw for names, element_type ONLY (NOT team)
    pr = players_raw_by_season.get(season)
    df = df.merge(
        pr[["element_type", "web_name", "first_name", "second_name"]],
        left_on="element", right_index=True, how="left"
    )

    # fixtures for true team mapping
    fx_path = DATA_ROOT / season / "fixtures.csv"
    fx = pd.read_csv(fx_path)
    
    # Identify Gameweek column
    gw_col = "event" if "event" in fx.columns else "round"
    
    # Convert and drop rows where Gameweek or Teams are missing
    fx[gw_col] = pd.to_numeric(fx[gw_col], errors="coerce")
    fx = fx.dropna(subset=[gw_col, "team_h", "team_a"])

    # reconstruct Player Team ID + Opponent Difficulty by matching opponent_team
    rows = []
    for _, r in fx.iterrows():
        # Using int() only after ensuring no NaNs exist in these specific rows
        gw = int(r[gw_col])
        h = int(r["team_h"])
        a = int(r["team_a"])
        dh = int(r["team_h_difficulty"]) if pd.notna(r["team_h_difficulty"]) else 0
        da = int(r["team_a_difficulty"]) if pd.notna(r["team_a_difficulty"]) else 0

        rows.append({"Gameweek": gw, "OppKey": a, "Player Team ID": h, "Opponent ID": a, "Is Home": True, "Opponent Difficulty": dh})
        rows.append({"Gameweek": gw, "OppKey": h, "Player Team ID": a, "Opponent ID": h, "Is Home": False, "Opponent Difficulty": da})

    opp_map = pd.DataFrame(rows)
    opp_map["Gameweek"] = opp_map["Gameweek"].astype("Int64")
    opp_map["OppKey"] = opp_map["OppKey"].astype("Int64")

    df = df.merge(
        opp_map,
        left_on=["Gameweek", "opponent_team"],
        right_on=["Gameweek", "OppKey"],
        how="left"
    ).drop(columns=["OppKey"])

    # team names
    teams_df = load_teams_for_season(season)

    df = df.merge(
        teams_df[["Team ID", "Team Name"]],
        left_on="Player Team ID",
        right_on="Team ID",
        how="left"
    ).rename(columns={"Team Name": "Player Team Name"}).drop(columns=["Team ID"])

    df = df.merge(
        teams_df.rename(columns={"Team ID": "Opponent ID", "Team Name": "Opponent Name"})[["Opponent ID", "Opponent Name"]],
        on="Opponent ID",
        how="left"
    )

    # clean names
    df["Player Name"] = (
        df["first_name"].fillna("") + " " + df["second_name"].fillna("")
    ).str.strip().replace("", np.nan).fillna(df["name"])
    df["Web Name"] = df["web_name"]

    return df

# Build the core frames for all selected seasons
core_frames = [build_season_core(s, df_gws[df_gws["season"] == s]) for s in sorted(df_gws["season"].unique())]
df_core = pd.concat(core_frames, ignore_index=True)

print("df_core rebuilt with 100% accurate Player Team IDs")
display(df_core.head())

Building df_core per season using fixture-based team identification...


Season 2020-21...
Season 2021-22...
Season 2022-23...
Season 2023-24...
Season 2024-25...
Season 2025-26...
df_core rebuilt with 100% accurate Player Team IDs


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,...,first_name,second_name,Player Team ID,Opponent ID,Is Home,Opponent Difficulty,Player Team Name,Opponent Name,Player Name,Web Name
0,Aaron Connolly,78,45,0,0,0,2,0,0,1,...,Aaron,Connolly,3.0,5.0,True,4.0,Brighton,Chelsea,Aaron Connolly,Connolly
1,Aaron Cresswell,435,90,0,0,0,2,0,0,1,...,Aaron,Cresswell,19.0,14.0,True,2.0,West Ham,Newcastle,Aaron Cresswell,Cresswell
2,Aaron Mooy,60,0,0,0,0,0,0,0,0,...,Aaron,Mooy,3.0,5.0,True,4.0,Brighton,Chelsea,Aaron Mooy,Mooy
3,Aaron Ramsdale,483,90,0,0,0,2,0,0,1,...,Aaron,Ramsdale,15.0,20.0,True,2.0,Sheffield Utd,Wolves,Aaron Ramsdale,Ramsdale
4,Abdoulaye Doucouré,512,90,0,0,1,0,0,0,3,...,Abdoulaye,Doucouré,7.0,17.0,False,4.0,Everton,Spurs,Abdoulaye Doucouré,Doucouré


In [9]:
# %%– Diagnostic: Opponent Difficulty completeness
print(" Opponent Difficulty — overall summary (df_core)")
total_rows = len(df_core)
missing_od = df_core["Opponent Difficulty"].isna().sum()
print(f"  Total rows: {total_rows:,}")
print(f"  Missing Opponent Difficulty: {missing_od:,} ({missing_od/total_rows*100:.4f}%)")

print("\n🔍 Missing Opponent Difficulty by season:")
season_stats = (
    df_core
    .groupby("season")["Opponent Difficulty"]
    .apply(lambda s: s.isna().sum())
    .to_frame("Missing_OD")
)
season_stats["Total_Rows"] = df_core.groupby("season")["Opponent Difficulty"].size()
season_stats["Missing_%"] = (season_stats["Missing_OD"] / season_stats["Total_Rows"] * 100).round(4)

display(season_stats)


 Opponent Difficulty — overall summary (df_core)
  Total rows: 174,307
  Missing Opponent Difficulty: 409 (0.2346%)

🔍 Missing Opponent Difficulty by season:


,Missing_OD,Total_Rows,Missing_%
season,,,
2020-21,0,27397,0.000
2021-22,0,29837,0.000
2022-23,0,29618,0.000
2023-24,0,31707,0.000
2024-25,0,28372,0.000
2025-26,409,27376,1.494


In [10]:
# Cleaning & canonical columns
POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

rename_map = {
    "element": "Code",
    "minutes": "Minutes Played",
    "goals_scored": "Goals Scored",
    "assists": "Assists",
    "clean_sheets": "Clean Sheet",
    "goals_conceded": "Goals Conceded",
    "yellow_cards": "Yellow Card",
    "red_cards": "Red Cards",
    "total_points": "Total Points",
    "influence": "Influence",
    "creativity": "Creativity",
    "threat": "Threat",
    "ict_index": "ICT Index",
}

df_clean = df_core.copy()
df_clean.rename(columns=rename_map, inplace=True)

# Position από element_type
if "element_type" in df_clean.columns:
    df_clean["Position"] = df_clean["element_type"].map(POS_MAP)
else:
    df_clean["Position"] = np.nan

# Normalized name
df_clean["Player Name Norm"] = df_clean["Player Name"].apply(normalize_player_name)

print("Sample cleaned rows:")
display(df_clean[[
    "Player Name", "Player Name Norm", "Player Team Name",
    "Opponent Name", "Total Points"
]].head())


Sample cleaned rows:


,Player Name,Player Name Norm,Player Team Name,Opponent Name,Total Points
0,Aaron Connolly,aaron connolly,Brighton,Chelsea,1
1,Aaron Cresswell,aaron cresswell,West Ham,Newcastle,1
2,Aaron Mooy,aaron mooy,Brighton,Chelsea,0
3,Aaron Ramsdale,aaron ramsdale,Sheffield Utd,Wolves,1
4,Abdoulaye Doucouré,abdoulaye doucoure,Everton,Spurs,3


In [12]:
# Injury flag (3+ consecutive 0 minutes) — VECTORIZED VERSION
df_clean = df_clean.sort_values(["Player Name Norm", "season", "Gameweek"])

def streak_flag(minutes_series):
    """Flag rows where player has had 0 minutes for 3+ consecutive gameweeks."""
    flags = []
    streak = 0
    for x in minutes_series:
        if x == 0:
            streak += 1
            flags.append(1 if streak >= 3 else 0)
        else:
            streak = 0
            flags.append(0)
    return flags

df_clean["Injury/Unavailable"] = (
    df_clean.groupby("Player Name Norm")["Minutes Played"]
    .transform(lambda s: streak_flag(s.values))
)

df_clean[["Player Name", "season", "Gameweek", "Minutes Played", "Injury/Unavailable"]].head(15)

,Player Name,season,Gameweek,Minutes Played,Injury/Unavailable
136097,Aaron Anselmino,2024-25,25,0,0
136908,Aaron Anselmino,2024-25,26,0,0
137691,Aaron Anselmino,2024-25,27,0,1
138479,Aaron Anselmino,2024-25,28,0,1
139137,Aaron Anselmino,2024-25,29,0,1
139909,Aaron Anselmino,2024-25,30,0,1
140701,Aaron Anselmino,2024-25,31,0,1
141694,Aaron Anselmino,2024-25,32,0,1
142918,Aaron Anselmino,2024-25,33,0,1
143644,Aaron Anselmino,2024-25,34,0,1


In [14]:
def add_lagged(df):
    metrics = [
        "Total Points", "Minutes Played", "Goals Scored", "Assists",
        "Goals Conceded", "ICT Index", "Threat", "Creativity", "Influence"
    ]

    df = df.sort_values(["Player Name Norm", "season", "Gameweek"]).copy()

    # Pre-filter to only existing columns once
    existing_metrics = [c for c in metrics if c in df.columns]

    for w in [3, 5]:
        # Shift all metrics at once per group, then compute rolling mean in bulk
        shifted = (
            df.groupby(["Player Name Norm", "season"])[existing_metrics]
            .shift(1)
        )
        rolled = shifted.groupby(
            [df["Player Name Norm"], df["season"]]
        ).transform(lambda s: s.rolling(w, min_periods=1).mean())

        for c in existing_metrics:
            df[f"Avg_{c}_L{w}"] = rolled[c].fillna(0)

    return df

df_lagged = add_lagged(df_clean)
display(df_lagged.head())

,name,Code,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Yellow Card,Red Cards,Total Points,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
136097,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136908,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137691,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138479,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
139137,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# UUID mapping (χρησιμοποιεί cleaned αν υπάρχει)
if CLEANED_UUID_MAPPING_PATH.exists():
    print(" Using cleaned UUID mapping:", CLEANED_UUID_MAPPING_PATH)
    mapping = pd.read_csv(CLEANED_UUID_MAPPING_PATH)

    norm_col = [c for c in mapping.columns if c.lower() == "player name norm"][0]
    uuid_col = [c for c in mapping.columns if c.lower() == "player uuid"][0]

    uuid_map = dict(zip(mapping[norm_col], mapping[uuid_col]))

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(uuid_map)

    print("UUIDs filled:", df_lagged["Player UUID"].notna().sum())

else:
    print("No cleaned mapping. Creating NEW stable mapping…")

    unique_norms = sorted(df_lagged["Player Name Norm"].unique())
    new_uuids = [str(uuid.uuid4()) for _ in unique_norms]

    mapping = pd.DataFrame({
        "Player Name Norm": unique_norms,
        "Player UUID": new_uuids
    })

    rep = df_lagged.groupby("Player Name Norm")[["Player Name", "Web Name"]] \
        .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan).reset_index()

    mapping = mapping.merge(rep, on="Player Name Norm", how="left")

    mapping.to_csv(UUID_MAPPING_PATH, index=False, encoding="utf-8-sig")
    print(f" Saved new UUID mapping → {UUID_MAPPING_PATH}")

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(
        dict(zip(mapping["Player Name Norm"], mapping["Player UUID"]))
    )

display(df_lagged[["Player Name","Player Name Norm","Player UUID"]].head())


No cleaned mapping. Creating NEW stable mapping…
 Saved new UUID mapping → output\player_uuid_mapping.csv


,Player Name,Player Name Norm,Player UUID
136097,Aaron Anselmino,aaron anselmino,1562eb35-1c75-4af9-9920-61ed2c7755c9
136908,Aaron Anselmino,aaron anselmino,1562eb35-1c75-4af9-9920-61ed2c7755c9
137691,Aaron Anselmino,aaron anselmino,1562eb35-1c75-4af9-9920-61ed2c7755c9
138479,Aaron Anselmino,aaron anselmino,1562eb35-1c75-4af9-9920-61ed2c7755c9
139137,Aaron Anselmino,aaron anselmino,1562eb35-1c75-4af9-9920-61ed2c7755c9


In [16]:
#Τελικό save σε training_data_v2.csv
base_cols = [
    "Player UUID", "Code", "Player Name", "Web Name", "Player Team Name",
    "season", "Gameweek", "Minutes Played", "Goals Scored", "Assists",
    "Clean Sheet", "Goals Conceded", "Yellow Card", "Red Cards",
    "Total Points", "Threat", "ICT Index", "Influence", "Creativity",
    "Opponent Name", "Opponent Difficulty", "Is Home", "Position",
    "Injury/Unavailable"
]

lagged_cols = [c for c in df_lagged.columns if c.startswith("Avg_")]
final_cols = base_cols + lagged_cols

df_final = df_lagged[final_cols].copy()

df_final.to_csv(TRAINING_PATH, index=False, encoding="utf-8-sig")

print(" training_data_v2.csv CREATED!")
print("Rows:", len(df_final))
print("Cols:", len(df_final.columns))
display(df_final.head())


 training_data_v2.csv CREATED!
Rows: 174307
Cols: 42


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
136097,1562eb35-1c75-4af9-9920-61ed2c7755c9,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,25,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136908,1562eb35-1c75-4af9-9920-61ed2c7755c9,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,26,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137691,1562eb35-1c75-4af9-9920-61ed2c7755c9,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,27,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138479,1562eb35-1c75-4af9-9920-61ed2c7755c9,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,28,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
139137,1562eb35-1c75-4af9-9920-61ed2c7755c9,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,29,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
#Sanity checks
print("Unique seasons:", sorted(df_final["season"].unique()))
print("Max GW per season:", df_final.groupby("season")["Gameweek"].max().to_dict())

print("\nOpponent Difficulty non-null:", df_final["Opponent Difficulty"].notna().sum())
display(
    df_final[
        ["Player Name","Player Team Name","Opponent Name","season","Gameweek","Opponent Difficulty"]
    ].head(20)
)


Unique seasons: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Max GW per season: {'2020-21': 38, '2021-22': 38, '2022-23': 38, '2023-24': 38, '2024-25': 38, '2025-26': 34}

Opponent Difficulty non-null: 173898


,Player Name,Player Team Name,Opponent Name,season,Gameweek,Opponent Difficulty
136097,Aaron Anselmino,Chelsea,Brighton,2024-25,25,3.0
136908,Aaron Anselmino,Chelsea,Aston Villa,2024-25,26,4.0
137691,Aaron Anselmino,Chelsea,Southampton,2024-25,27,1.0
138479,Aaron Anselmino,Chelsea,Leicester,2024-25,28,1.0
139137,Aaron Anselmino,Chelsea,Arsenal,2024-25,29,5.0
139909,Aaron Anselmino,Chelsea,Spurs,2024-25,30,2.0
140701,Aaron Anselmino,Chelsea,Brentford,2024-25,31,3.0
141694,Aaron Anselmino,Chelsea,Ipswich,2024-25,32,2.0
142918,Aaron Anselmino,Chelsea,Fulham,2024-25,33,3.0
143644,Aaron Anselmino,Chelsea,Everton,2024-25,34,3.0


In [18]:
print("SAFETY CHECK — Comparing column structure before merge")

OLD_TRAIN = "output/training_data_v2.csv"
NEW_TRAIN = "output/training_data_2025_26.csv"

df_old = pd.read_csv(OLD_TRAIN, encoding="utf-8-sig")
df_new = pd.read_csv(NEW_TRAIN, encoding="utf-8-sig")

print(f"   → Old dataset rows: {len(df_old):,}")
print(f"   → New dataset rows: {len(df_new):,}")

cols_old = set(df_old.columns)
cols_new = set(df_new.columns)

missing_in_new = cols_old - cols_new
missing_in_old = cols_new - cols_old

print("\n Columns missing in NEW dataset:", missing_in_new)
print(" Columns missing in OLD dataset:", missing_in_old)



SAFETY CHECK — Comparing column structure before merge


   → Old dataset rows: 174,307
   → New dataset rows: 27,376

 Columns missing in NEW dataset: {'Avg_ICT Index_L3', 'Avg_ICT Index_L5', 'Avg_Influence_L5', 'Avg_Influence_L3', 'Avg_Threat_L5', 'Player Team Name', 'Threat', 'Yellow Card', 'Avg_Creativity_L5', 'Avg_Goals Conceded_L5', 'Influence', 'Avg_Creativity_L3', 'Avg_Goals Conceded_L3', 'Web Name', 'Red Cards', 'Player Name', 'Code', 'Opponent Name', 'Creativity', 'Avg_Threat_L3'}
 Columns missing in OLD dataset: {'name', 'was_home', 'bps', 'expected_assists', 'element', 'saves', 'Player Name Norm', 'element_type', 'Player Team ID', 'bonus', 'opponent_team', 'penalties_saved', 'yellow_cards', 'recoveries', 'clearances_blocks_interceptions', 'penalties_missed', 'starts', 'in_dreamteam', 'creativity', 'expected_goals_conceded', 'tackles', 'expected_goal_involvements', 'influence', 'red_cards', 'team', 'defensive_contribution', 'expected_goals', 'own_goals', 'threat', 'Team_Points_Contribution_GW_Pct'}


In [19]:

# SAFE MERGE — KEEP EXACT COLUMN ORDER FROM training_data_v2.csv

print(" Loading old + new datasets with SAFE column order rules...")

OLD_TRAIN = "output/training_data_v2.csv"
NEW_TRAIN = "output/training_data_2025_26.csv"
MERGED_OUT = "output/training_data_v3.csv"

df_old = pd.read_csv(OLD_TRAIN, encoding="utf-8-sig")
df_new = pd.read_csv(NEW_TRAIN, encoding="utf-8-sig")

print(f"   → Old dataset rows: {len(df_old):,}")
print(f"   → New dataset rows: {len(df_new):,}")


#Keep only the columns of the OLD file
# (the new dataset will be forced to follow this order)

old_cols = list(df_old.columns)

# Keep only the columns that exist in both
common_cols = [c for c in old_cols if c in df_new.columns]

print(f"Keeping {len(common_cols)} common columns in correct order")

df_old_clean = df_old[common_cols]
df_new_clean = df_new[common_cols]

# -----------------------------------------
# MERGE with correct column order
# -----------------------------------------
df_merged = pd.concat([df_old_clean, df_new_clean], ignore_index=True)

print(f" MERGED successfully → {len(df_merged):,} rows total")

# -----------------------------------------
# SAVE FINAL
# -----------------------------------------
df_merged.to_csv(MERGED_OUT, index=False, encoding="utf-8-sig")
print(f" Saved → {MERGED_OUT}")

# -----------------------------------------
#  VERIFY
# -----------------------------------------
print("\n Sanity check:")
print("Columns:", list(df_merged.columns))


 Loading old + new datasets with SAFE column order rules...
   → Old dataset rows: 174,307
   → New dataset rows: 27,376
Keeping 22 common columns in correct order
 MERGED successfully → 201,683 rows total
 Saved → output/training_data_v3.csv

 Sanity check:
Columns: ['Player UUID', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Total Points', 'ICT Index', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5']


In [20]:
print(" Checking duplicates (season + GW + Player UUID)...")

dups = df_merged[df_merged.duplicated(
    subset=["Player UUID", "season", "Gameweek"],
    keep=False
)]

print(f"Found {len(dups)} duplicated rows")
display(dups.head(20))


 Checking duplicates (season + GW + Player UUID)...
Found 34844 duplicated rows


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Position,Injury/Unavailable,Avg_Total Points_L3,Avg_Minutes Played_L3,Avg_Goals Scored_L3,Avg_Assists_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5
46,1562eb35-1c75-4af9-9920-61ed2c7755c9,2025-26,33,0,0,0,0,0,0,0.0,...,DEF,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
47,1562eb35-1c75-4af9-9920-61ed2c7755c9,2025-26,33,0,0,0,0,0,0,0.0,...,DEF,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
48,1562eb35-1c75-4af9-9920-61ed2c7755c9,2025-26,33,0,0,0,0,0,0,0.0,...,DEF,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
75,a4be9652-b8b9-4ab0-94d5-406007523e66,2020-21,26,60,0,0,0,1,2,4.0,...,FWD,0.0,0.666667,20.333333,0.0,0.0,0.8,15.8,0.0,0.0
76,a4be9652-b8b9-4ab0-94d5-406007523e66,2020-21,26,60,0,0,0,1,2,4.0,...,FWD,0.0,0.666667,20.000000,0.0,0.0,1.0,25.6,0.0,0.0
108,a4be9652-b8b9-4ab0-94d5-406007523e66,2021-22,22,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.0,0.6,19.0,0.0,0.0
109,a4be9652-b8b9-4ab0-94d5-406007523e66,2021-22,22,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.0,0.4,12.0,0.0,0.0
110,a4be9652-b8b9-4ab0-94d5-406007523e66,2021-22,22,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
112,a4be9652-b8b9-4ab0-94d5-406007523e66,2021-22,25,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
113,a4be9652-b8b9-4ab0-94d5-406007523e66,2021-22,25,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
df_merged = df_merged.drop_duplicates(
    subset=["Player UUID", "season", "Gameweek"],
    keep="first"
).reset_index(drop=True)

print(" Duplicates removed. New shape:", df_merged.shape)


 Duplicates removed. New shape: (179756, 22)


In [22]:
dups.groupby(["Player UUID","season","Gameweek"]).size().sort_values(ascending=False).head(20)




Player UUID                           season   Gameweek
1496450a-a7f0-4f5c-9e59-e47edadf2ac3  2020-21  26          8
                                      2021-22  36          8
                                               29          7
063a4c9b-6593-4cf5-82aa-f5ea2917c01e  2020-21  35          6
b2d347f5-ad33-4336-9c3c-a115509cf556  2020-21  35          6
a1978787-aaba-4d9f-9307-740f8bfd493a  2020-21  35          6
5790d043-948c-4a5f-825d-9ca93fa8364f  2020-21  35          6
fe955101-c0cf-42ce-b57e-5a5959b677a4  2020-21  35          6
977298f0-c97f-4c03-b76d-0b57ea19f1d6  2020-21  35          6
0ecb6cf5-f7e0-4e31-8a73-e7ed09d467fb  2020-21  35          6
98eeae45-ac2a-478f-8eb9-39ac53824f00  2020-21  35          6
09ccd89d-bfe2-4831-ac58-a057c81ec72d  2020-21  35          6
f863e661-0d88-4ec7-9eeb-068c5ce61081  2020-21  35          6
12bc5539-d337-40e1-9dbd-b3fc834c0cce  2020-21  35          6
9b877637-f1bd-41dd-8641-e0cdb11f0145  2020-21  35          6
0a544157-4f52-483f-9ed6-c0121

In [23]:
df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

dups = df[df.duplicated(["Player UUID", "season", "Gameweek"], keep=False)]
dups.sort_values(["Player UUID","season","Gameweek"]).head(200)


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Position,Injury/Unavailable,Avg_Total Points_L3,Avg_Minutes Played_L3,Avg_Goals Scored_L3,Avg_Assists_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5
42010,0019d63f-464e-43af-9819-1496d0cf4c33,2022-23,25,70,0,0,1,0,2,4.4,...,FWD,0.0,0.666667,17.000000,0.0,0.000000,0.4,10.2,0.0,0.0
42011,0019d63f-464e-43af-9819-1496d0cf4c33,2022-23,25,75,0,1,1,0,5,5.6,...,FWD,0.0,1.333333,40.333333,0.0,0.000000,0.8,24.2,0.0,0.0
42012,0019d63f-464e-43af-9819-1496d0cf4c33,2022-23,25,75,0,1,1,0,5,5.6,...,FWD,0.0,2.666667,58.666667,0.0,0.333333,1.8,39.2,0.0,0.2
42015,0019d63f-464e-43af-9819-1496d0cf4c33,2022-23,29,69,0,1,0,3,5,5.2,...,FWD,0.0,2.333333,44.000000,0.0,0.333333,2.8,55.4,0.0,0.4
42016,0019d63f-464e-43af-9819-1496d0cf4c33,2022-23,29,90,0,0,1,0,2,1.4,...,FWD,0.0,2.333333,42.000000,0.0,0.333333,3.4,55.2,0.0,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35061,016166b9-170b-42b4-ae49-f4dc6bc309e7,2021-22,29,0,0,0,0,0,0,0.0,...,FWD,1.0,0.333333,1.000000,0.0,0.000000,0.2,0.6,0.0,0.0
35062,016166b9-170b-42b4-ae49-f4dc6bc309e7,2021-22,29,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.000000,0.2,0.6,0.0,0.0
35066,016166b9-170b-42b4-ae49-f4dc6bc309e7,2021-22,33,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
35067,016166b9-170b-42b4-ae49-f4dc6bc309e7,2021-22,33,0,0,0,0,0,0,0.0,...,FWD,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0


In [24]:

import pandas as pd
from pathlib import Path

# 1. Define paths
TRAIN_FILE = Path("output/training_data_v3.csv")
MAPPING_FILE = Path("output/player_uuid_mapping.csv")

# 2. Load the training data and the mapping
df = pd.read_csv(TRAIN_FILE, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_FILE, encoding="utf-8-sig")

# 3. Find the UUID for Zidane Iqbal from the mapping file
# We search for "Zidane" in the 'Player Name Norm' or 'Player Name' columns of the mapping
iqbal_mapping = mapping[mapping["Player Name Norm"].str.contains("zidane", case=False, na=False)]

if not iqbal_mapping.empty:
    iqbal_uuid = iqbal_mapping["Player UUID"].iloc[0]
    print(f"Found UUID for Zidane Iqbal: {iqbal_uuid}")
    
    # 4. Filter the training data using this UUID and the season
    sub = df[
        (df["Player UUID"] == iqbal_uuid) & 
        (df["season"] == "2022-23")
    ]
    
    # 5. Save and Print
    if not sub.empty:
        sub.to_csv("output/iqbal_2022_23.csv", index=False, encoding="utf-8-sig")
        print(f"Found {len(sub)} rows for Zidane Iqbal in 2022-23.")
        print(sub)
    else:
        print("No match found for that UUID in the 2022-23 season data.")
else:
    print("Could not find Zidane Iqbal in the player_uuid_mapping.csv file.")

Found UUID for Zidane Iqbal: 7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2
Found 46 rows for Zidane Iqbal in 2022-23.
                                 Player UUID   season  Gameweek  \
174261  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23         1   
174262  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23         2   
174263  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23         3   
174264  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23         4   
174265  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23         5   
174266  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23         6   
174267  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23         9   
174268  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23        10   
174269  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23        11   
174270  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23        12   
174271  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23        13   
174272  7ac117c2-01dc-4a8b-9ff6-33c5570a1fb2  2022-23        14   
174273  7ac117c2-01d

In [25]:
print("🧹 Fixing duplicate rows (Player UUID + season + GW)...")

df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

# Step 1 – Προτεραιότητα στα πραγματικά ματς (Minutes > 0)
df["PlayedFlag"] = df["Minutes Played"] > 0

# Step 2 – Βαθμολογούμε rows ώστε να ξέρουμε ποιο να κρατήσουμε
df["keep_rank"] = df.groupby(
    ["Player UUID", "season", "Gameweek"]
)["PlayedFlag"].transform(lambda s: s.rank(method="first", ascending=False))

# Step 3 – Κρατάμε ΜΟΝΟ το 1 καλύτερο
df_dedup = df[df["keep_rank"] == 1].drop(columns=["PlayedFlag", "keep_rank"])

print("✔️ Before:", len(df))
print("✔️ After :", len(df_dedup))
print("✔️ Removed duplicates:", len(df) - len(df_dedup))

df_dedup.to_csv("output/training_data_v3_clean.csv", index=False, encoding="utf-8-sig")
print("💾 Saved cleaned → training_data_v3_clean.csv")


🧹 Fixing duplicate rows (Player UUID + season + GW)...


KeyboardInterrupt: 

In [26]:
print("🧹 Fixing duplicate rows (Player UUID + season + GW)...")
df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

print("✔️ Before:", len(df))

# Sort so that rows with Minutes > 0 come first, then keep the first per group
df_dedup = (
    df.sort_values(
        ["Player UUID", "season", "Gameweek", "Minutes Played"],
        ascending=[True, True, True, False]  # descending Minutes = played rows first
    )
    .drop_duplicates(subset=["Player UUID", "season", "Gameweek"], keep="first")
    .reset_index(drop=True)
)

print("✔️ After :", len(df_dedup))
print("✔️ Removed duplicates:", len(df) - len(df_dedup))

df_dedup.to_csv("output/training_data_v3_clean.csv", index=False, encoding="utf-8-sig")
print("💾 Saved cleaned → training_data_v3_clean.csv")

🧹 Fixing duplicate rows (Player UUID + season + GW)...
✔️ Before: 201683
✔️ After : 179756
✔️ Removed duplicates: 21927
💾 Saved cleaned → training_data_v3_clean.csv


In [27]:
# Cell: FINAL SAFE VALIDATION (CLEAN VERSION)

import pandas as pd
import os

print("Loading cleaned training dataset...")
df = pd.read_csv("output/training_data_v3_clean.csv", encoding="utf-8-sig")
print(f"Rows: {len(df):,}")
print(f"Available Columns: {df.columns.tolist()}")

# 1. DUPLICATES CHECK
print("Checking duplicates (Player UUID + season + GW)...")
if "Player UUID" in df.columns:
    dupes = df.duplicated(subset=["Player UUID","season","Gameweek"], keep=False)
    dupe_rows = df[dupes]
    print(f"Duplicate rows found: {len(dupe_rows):,}")
    if len(dupe_rows) > 0:
        display(dupe_rows.head(20))
    else:
        print("No duplicates detected.")
else:
    print("Skip: 'Player UUID' not found.")

# 2. MISSING OPPONENT DIFFICULTY
print("Checking missing Opponent Difficulty...")
if "Opponent Difficulty" in df.columns:
    missing_od = df[df["Opponent Difficulty"].isna()]
    print(f"Missing OD count: {len(missing_od):,}")
    if len(missing_od) > 0:
        display(missing_od.head(20))
    else:
        print("No missing OD values.")
else:
    print("Skip: 'Opponent Difficulty' not found.")

# 3. GAMEWEEK GAPS
print("Checking gameweek gaps per player...")
if "Player UUID" in df.columns:
    gap_examples = []
    # Check a sample of 100 players
    sample_pids = df["Player UUID"].unique()[:100] 
    for pid in sample_pids:
        for season in df[df["Player UUID"] == pid]["season"].unique():
            sub = df[(df["Player UUID"] == pid) & (df["season"] == season)]
            gws = sorted(sub["Gameweek"].unique())
            expected = list(range(min(gws), max(gws)+1))
            if gws != expected:
                gap_examples.append((pid, season, gws))
                
    print(f"Players with gaps in sample: {len(gap_examples)}")
    print("Note: Gaps are expected as players may miss matches.")
else:
    print("Skip: 'Player UUID' not found.")

# 4. TEAM NAME CONSISTENCY CHECK
print("Checking team name consistency...")
if "Player Team Name" in df.columns:
    team_file = "data/2022-23/teams.csv"
    if os.path.exists(team_file):
        teams_df = pd.read_csv(team_file, encoding="utf-8-sig")
        name_col = "name" if "name" in teams_df.columns else teams_df.columns[0]
        unique_team_names = sorted(teams_df[name_col].unique())

        invalid_teams = df[~df["Player Team Name"].isin(unique_team_names)]
        print(f"Invalid team names: {len(invalid_teams):,}")
        if len(invalid_teams) > 0:
            display(invalid_teams.head(20))
        else:
            print("All team names match known Premier League teams.")
    else:
        print(f"Skip: Reference file {team_file} not found.")
else:
    print("Info: 'Player Team Name' not in this CSV. Skipping team check.")

# 5. OPPONENT NAME CONSISTENCY
print("Checking opponent name consistency...")
if "Opponent Name" in df.columns:
    invalid_opps = df[~df["Opponent Name"].isin(unique_team_names)]
    print(f"Invalid opponent names: {len(invalid_opps):,}")
    if len(invalid_opps) > 0:
        display(invalid_opps.head(20))
    else:
        print("All opponent names valid.")
else:
    print("Info: 'Opponent Name' not in this CSV. Skipping opponent check.")

print("VALIDATION COMPLETE")

Loading cleaned training dataset...
Rows: 179,756
Available Columns: ['Player UUID', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Total Points', 'ICT Index', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5']
Checking duplicates (Player UUID + season + GW)...
Duplicate rows found: 0
No duplicates detected.
Checking missing Opponent Difficulty...
Missing OD count: 818


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Position,Injury/Unavailable,Avg_Total Points_L3,Avg_Minutes Played_L3,Avg_Goals Scored_L3,Avg_Assists_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5
355,005b9792-b2bb-4150-901e-cb310ac1c0da,2025-26,34,0,0,0,0,0,0,0.0,...,DEF,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
568,008b487c-8aa1-4bb8-a2d7-b1436df5e706,2025-26,31,0,0,0,0,0,0,0.0,...,MID,0.0,0.666667,28.333333,0.0,0.000000,1.4,52.0,0.0,0.0
605,008dd810-cdd1-4fab-ab6f-c198ed5550f9,2025-26,34,0,0,0,0,0,0,0.0,...,FWD,0.0,1.000000,18.000000,0.0,0.000000,0.6,10.8,0.0,0.0
857,010886bd-a6e9-43d7-b289-84104630c148,2025-26,34,0,0,0,0,0,0,0.0,...,DEF,0.0,5.000000,180.000000,0.0,0.000000,3.4,144.0,0.0,0.0
898,0121f150-0771-44f9-b792-27fc6fd3a253,2025-26,31,0,0,0,0,0,0,0.0,...,MID,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
1182,0176c2f9-825d-48dc-8b72-2a9c8fb06946,2025-26,31,0,0,0,0,0,0,0.0,...,MID,1.0,0.000000,0.000000,0.0,0.000000,1.2,1.2,0.2,0.0
1398,01e14bee-8508-4fbc-bb7f-320acd6c337c,2025-26,31,0,0,0,0,0,0,0.0,...,MID,0.0,2.000000,40.666667,0.0,0.333333,2.0,47.0,0.0,0.2
1627,02b16dd5-68a4-4cbc-adc9-50767637ea97,2025-26,31,0,0,0,0,0,0,0.0,...,MID,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
1728,02dae0e5-5135-4ca1-9a87-7f1cf491f2b3,2025-26,34,0,0,0,0,0,0,0.0,...,DEF,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
2208,034c7bb2-ab76-4465-9fda-7f2b8fe08e7d,2025-26,34,0,0,0,0,0,0,0.0,...,MID,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0


Checking gameweek gaps per player...
Players with gaps in sample: 129
Note: Gaps are expected as players may miss matches.
Checking team name consistency...
Info: 'Player Team Name' not in this CSV. Skipping team check.
Checking opponent name consistency...
Info: 'Opponent Name' not in this CSV. Skipping opponent check.
VALIDATION COMPLETE


In [28]:
import numpy as np
import pandas as pd
from pathlib import Path
import re
from unidecode import unidecode

def normalize_player_name(name):
    if pd.isna(name): return ""
    name = str(name).strip().lower()
    name = unidecode(name)
    name = re.sub(r"[\s_]*\d+\s*$", "", name)
    name = name.replace("_", " ")
    name = "".join(c for c in name if c.isalnum() or c.isspace())
    return " ".join(name.split())

print("Loading datasets...")
df = pd.read_csv("output/training_data_v3_clean.csv", encoding="utf-8-sig")
mapping = pd.read_csv("output/player_uuid_mapping.csv")

# 1. Build a Season-Agnostic Bridge for 2025-26
print("Building specialized 2025-26 Bridge...")
pr_2025_path = Path("data/2025-26/players_raw.csv")
if not pr_2025_path.exists():
    raise FileNotFoundError("Could not find data/2025-26/players_raw.csv")

pr_2025 = pd.read_csv(pr_2025_path)
id_col = "element" if "element" in pr_2025.columns else "id"

# Create normalization for 2025-26 specifically
pr_2025["norm_full"] = (pr_2025["first_name"].fillna("") + " " + pr_2025["second_name"].fillna("")).apply(normalize_player_name)
pr_2025["norm_web"] = pr_2025["web_name"].fillna("").apply(normalize_player_name)

# Create a lookup dictionary for 2025-26: Name -> (Code, Team)
lookup_2025 = {}
for _, r in pr_2025.iterrows():
    lookup_2025[r["norm_full"]] = (r[id_col], r["team"])
    lookup_2025[r["norm_web"]] = (r[id_col], r["team"])

# 2. Build Historical Bridge (2020-2025)
print("Building Historical Bridge...")
bridge_list = []
for season_folder in Path("data").glob("20*"):
    if season_folder.name == "2025-26": continue
    pr_path = season_folder / "players_raw.csv"
    if pr_path.exists():
        pr = pd.read_csv(pr_path)
        cid = "id" if "id" in pr.columns else "element"
        pr["norm_full"] = (pr["first_name"].fillna("") + " " + pr["second_name"].fillna("")).apply(normalize_player_name)
        temp = pr[["norm_full", cid, "team"]].copy()
        temp.columns = ["Player Name Norm", "Code", "Player Team ID"]
        temp["season"] = season_folder.name
        bridge_list.append(temp)

hist_bridge = pd.concat(bridge_list).drop_duplicates(["Player Name Norm", "season"])

# 3. Apply Links to Training Data
print("Applying Links...")
if "Player Name Norm" not in mapping.columns:
    mapping["Player Name Norm"] = mapping["Player Name"].apply(normalize_player_name)
if "Web Name" in mapping.columns:
    mapping["Web Name Norm"] = mapping["Web Name"].apply(normalize_player_name)
else:
    mapping["Web Name Norm"] = mapping["Player Name Norm"]

# Map UUID to Names
df = df.merge(mapping[["Player UUID", "Player Name Norm", "Web Name Norm"]], on="Player UUID", how="left")

# Link 2025-26 rows using the specialized lookup
mask_2025 = df["season"] == "2025-26"
def get_2025_data(row):
    # Try full name then web name
    res = lookup_2025.get(row["Player Name Norm"])
    if not res:
        res = lookup_2025.get(row["Web Name Norm"])
    return pd.Series(res) if res else pd.Series([np.nan, np.nan])

print("Mapping 2025-26 data specifically...")
df.loc[mask_2025, ["Code", "Player Team ID"]] = df[mask_2025].apply(get_2025_data, axis=1).values

# Link Historical rows
print("Mapping Historical data...")
df = df.merge(hist_bridge, on=["Player Name Norm", "season"], how="left", suffixes=("", "_hist"))
df["Code"] = df["Code"].fillna(df["Code_hist"])
df["Player Team ID"] = df["Player Team ID"].fillna(df["Player Team ID_hist"])
df.drop(columns=["Code_hist", "Player Team ID_hist"], inplace=True)

# Final Check
missing_count = df["Player Team ID"].isna().sum()
if missing_count > 0:
    print(f"Missing {missing_count} rows.")
    print(df[df["Player Team ID"].isna()]["season"].value_counts())
    df["Player Team ID"] = df["Player Team ID"].fillna(0).astype(int)
else:
    print("Success: 100% of rows matched!")

# 4. Team Contributions
print("Calculating Team Contributions...")
team_stats = df.groupby(["Player Team ID", "season", "Gameweek"])["Total Points"].sum().reset_index()
team_stats.rename(columns={"Total Points": "Team_Total_Points_GW"}, inplace=True)
team_stats["Team_Total_Points_CUM"] = team_stats.groupby(["Player Team ID", "season"])["Team_Total_Points_GW"].cumsum()
team_stats["Team_Total_Points"] = team_stats.groupby(["Player Team ID", "season"])["Team_Total_Points_CUM"].shift(1).fillna(0)

df = df.merge(team_stats, on=["Player Team ID", "season", "Gameweek"], how="left")
df["Player_Season_Points"] = df.groupby(["Player UUID", "season"])["Total Points"].cumsum().shift(1).fillna(0)

df["Team_Points_Contribution_GW_Pct"] = np.where(df["Team_Total_Points_GW"] > 0, (df["Total Points"] / df["Team_Total_Points_GW"] * 100), 0).round(2)
df["Team_Points_Contribution_Causal_Pct"] = np.where(df["Team_Total_Points"] > 0, (df["Player_Season_Points"] / df["Team_Total_Points"] * 100), 0).round(2)

df.to_csv("output/training_data_v4_with_contrib.csv", index=False, encoding="utf-8-sig")
print("DONE!")

Loading datasets...
Building specialized 2025-26 Bridge...
Building Historical Bridge...
Applying Links...
Mapping 2025-26 data specifically...
Mapping Historical data...
Missing 26396 rows.
season
2025-26    26396
Name: count, dtype: int64
Calculating Team Contributions...
DONE!


In [29]:
# Check the current season's raw file
pr_2025 = pd.read_csv("data/2025-26/players_raw.csv")
print("Columns in 2025-26 players_raw:", pr_2025.columns.tolist())
print("\nFirst 3 rows of 2025-26 data:")
print(pr_2025.head(3))

Columns in 2025-26 players_raw: ['element', 'team', 'element_type', 'first_name', 'second_name', 'web_name']

First 3 rows of 2025-26 data:
   element  team  element_type first_name            second_name      web_name
0        1     1             1      David            Raya Martín          Raya
1        2     1             1       Kepa  Arrizabalaga Revuelta  Arrizabalaga
2        3     1             1       Karl                   Hein          Hein


In [30]:

# Load the file with low_memory=False to handle potential mixed types in index columns
df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig", low_memory=False)

print("Loaded:", len(df), "rows")

# Columns produced by the v4 Golden Cell
required_cols = [
    "Team_Total_Points_GW", "Team_Total_Points", "Player_Season_Points",
    "Team_Points_Contribution_GW_Pct", "Team_Points_Contribution_Causal_Pct",
    "Team_Contribution_Rank_GW"
]

print("Checking required columns...")
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print("Missing columns:", missing)
    print("Available columns are:", df.columns.tolist())
else:
    print("All contribution columns present.")

# Check for NaNs
print("Checking NaNs in contribution cols...")
existing_req = [c for c in required_cols if c in df.columns]
nan_counts = df[existing_req].isna().sum()
if nan_counts.sum() > 0:
    print(nan_counts[nan_counts > 0])
else:
    print("No NaNs detected.")

# Check GW1 causal logic
print("Checking Gameweek 1 causal values (Expect 0)...")
gw1 = df[df["Gameweek"] == 1]
print("Team_Total_Points > 0 on GW1:", (gw1["Team_Total_Points"] > 0).sum())
print("Player_Season_Points > 0 on GW1:", (gw1["Player_Season_Points"] > 0).sum())

# Check sum of GW contributions
team_col = "Player Team ID" if "Player Team ID" in df.columns else "team"
if team_col in df.columns:
    print(f"Checking if GW contributions sum to ~100% per team/week (using {team_col})...")
    group_sum = df.groupby([team_col, "season", "Gameweek"])["Team_Points_Contribution_GW_Pct"].sum()
    group_sum = group_sum[group_sum > 0]
    bad_groups = group_sum[(group_sum < 99.9) | (group_sum > 100.1)]
    print("Groups failing 100% sum check:", len(bad_groups))
    if len(bad_groups) > 0:
        print(bad_groups.head())
else:
    print("Skip: Team identifier column not found.")

# Summary statistics
print("Contribution percentage summary:")
display(df[["Team_Points_Contribution_GW_Pct", "Team_Points_Contribution_Causal_Pct"]].describe())

print("Top 10 Season-Long Contributors (Causal Pct):")
display(df.sort_values("Team_Points_Contribution_Causal_Pct", ascending=False).head(10))

Loaded: 179756 rows
Checking required columns...
Missing columns: ['Team_Contribution_Rank_GW']
Available columns are: ['Player UUID', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Total Points', 'ICT Index', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5', 'Player Name Norm', 'Web Name Norm', 'Code', 'Player Team ID', 'Team_Total_Points_GW', 'Team_Total_Points_CUM', 'Team_Total_Points', 'Player_Season_Points', 'Team_Points_Contribution_GW_Pct', 'Team_Points_Contribution_Causal_Pct']
Checking NaNs in contribution cols...
No NaNs detected.
Checking Gameweek 1 causal values (Expect 0)...
Team_Total_Points > 0 on GW1: 0
Player_Season_Points > 0 on GW1: 3057
Checking if GW contributions sum to ~100% per team/week (using Player Team ID)...
Gro

,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct
count,179756.000000,179756.000000
mean,2.412701,2.849179
std,5.089566,67.449423
min,-150.000000,-11.760000
25%,0.000000,0.000000
50%,0.000000,0.420000
75%,3.330000,4.170000
max,100.000000,16600.000000


Top 10 Season-Long Contributors (Causal Pct):


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Player Name Norm,Web Name Norm,Code,Player Team ID,Team_Total_Points_GW,Team_Total_Points_CUM,Team_Total_Points,Player_Season_Points,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct
22857,1bfbe743-e8d2-4607-a070-21b785b3c0c5,2020-21,2,0,0,0,0,0,0,0.0,...,henri lansbury,lansbury,30.0,2,66,67,1.0,166.0,0.00,16600.0
170592,f2c93ee3-82ab-4a3a-be29-2df82c2ad422,2020-21,2,0,0,0,0,0,0,0.0,...,tom heaton,heaton,28.0,2,66,67,1.0,128.0,0.00,12800.0
86551,7af251a1-4a6b-4337-83ab-3ba786e92e4c,2020-21,2,0,0,0,0,0,0,0.0,...,jose ignacio peleteiro romallo,jota,34.0,2,66,67,1.0,118.0,0.00,11800.0
127925,b80179fe-2e0b-4f3c-8620-0c043408b3f2,2020-21,2,0,0,0,0,0,0,0.0,...,kortney hause,hause,39.0,2,66,67,1.0,102.0,0.00,10200.0
123528,b12a74ce-fd85-47fe-9335-914d9ffe0b6f,2020-21,2,90,0,0,1,0,2,5.6,...,ollie watkins,watkins,514.0,2,66,67,1.0,91.0,3.03,9100.0
5701,07c124e5-1547-4a4d-8138-2637b28d8fa8,2020-21,2,28,0,0,0,0,1,2.3,...,keinan davis,davis,50.0,2,66,67,1.0,44.0,1.52,4400.0
141292,ca0f681a-cbbf-4b57-8b4d-375777f0e88e,2020-21,2,90,0,0,1,0,3,5.9,...,douglas luiz soares de paulo,douglas luiz,52.0,2,66,67,1.0,42.0,4.55,4200.0
88276,7dd59030-0adc-4186-99c5-d6d1723e43bc,2020-21,2,61,0,0,1,0,3,4.7,...,conor hourihane,hourihane,33.0,2,66,67,1.0,31.0,4.55,3100.0
35111,309f7344-e1cf-4260-86b7-a9c8f444c0ce,2020-21,2,90,0,0,1,0,3,9.7,...,jack grealish,grealish,37.0,2,66,67,1.0,7.0,4.55,700.0
75821,696d1b58-ead4-4a2a-af67-90896e49b161,2023-24,2,0,0,0,0,0,0,0.0,...,jack harrison,harrison,661.0,9,8,28,20.0,130.0,0.00,650.0


In [31]:
# Load the current dataset
file_path = "output/training_data_v4_with_contrib.csv"
df = pd.read_csv(file_path, low_memory=False)

print(f"Starting cleanup on {len(df)} rows...")

# 1. Force Causal/Historical features to 0 for Gameweek 1
# This ensures a clean start for every season
causal_cols = ["Team_Total_Points", "Player_Season_Points", "Team_Points_Contribution_Causal_Pct"]

for col in causal_cols:
    if col in df.columns:
        df.loc[df["Gameweek"] == 1, col] = 0

# 2. Re-calculate Team_Contribution_Rank_GW
# We rank players within their team for each gameweek (1.0 = top contributor)
if "Team_Points_Contribution_GW_Pct" in df.columns:
    print("Calculating Team_Contribution_Rank_GW...")
    df["Team_Contribution_Rank_GW"] = df.groupby(["Player Team ID", "season", "Gameweek"])["Team_Points_Contribution_GW_Pct"].rank(
        method="min", 
        ascending=False, 
        pct=True
    ).round(4)

# 3. Clean up Team ID 0 artifacts
# If a player is unmatched (Team ID 0), we set their contribution metrics to 0
mask_unmatched = df["Player Team ID"] == 0
df.loc[mask_unmatched, ["Team_Points_Contribution_GW_Pct", "Team_Contribution_Rank_GW"]] = 0

# 4. Final Formatting
# Ensure all contribution columns are rounded for cleanliness
cols_to_round = ["Team_Points_Contribution_GW_Pct", "Team_Points_Contribution_Causal_Pct"]
for col in cols_to_round:
    if col in df.columns:
        df[col] = df[col].round(2)

# Save the polished version
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print("Cleanup complete!")
print("Causal GW1 values reset to 0.")
print("Team_Contribution_Rank_GW has been added.")
print(f"File saved to: {file_path}")

Starting cleanup on 179756 rows...
Calculating Team_Contribution_Rank_GW...
Cleanup complete!
Causal GW1 values reset to 0.
Team_Contribution_Rank_GW has been added.
File saved to: output/training_data_v4_with_contrib.csv


In [32]:
# Load the dataset
file_path = "output/training_data_v4_with_contrib.csv"
df = pd.read_csv(file_path, low_memory=False)

print(f"Initial row count: {len(df):,}")

# 1. Remove rows where Position is missing (mostly Managers)
df_clean = df.dropna(subset=['Position']).copy()

# 2. Convert Position to categorical to save memory and prepare for ML
df_clean['Position'] = df_clean['Position'].astype('category')

print(f"Rows removed: {len(df) - len(df_clean)}")
print(f"Final clean row count: {len(df_clean):,}")

# 3. Final check: Are there any other NaNs?
nan_summary = df_clean.isna().sum()
cols_with_nans = nan_summary[nan_summary > 0]

if not cols_with_nans.empty:
    print("Remaining columns with NaNs:")
    print(cols_with_nans)
else:
    print("Success: No missing values remain in the dataset.")

# Save the finalized training set
df_clean.to_csv("output/final_training_data_v5.csv", index=False, encoding="utf-8-sig")
print("Saved to: output/final_training_data_v5.csv")

Initial row count: 179,756
Rows removed: 312
Final clean row count: 179,444
Remaining columns with NaNs:
Opponent Difficulty      818
Is Home                  818
Player Name Norm       26396
Web Name Norm          26396
Code                   26396
dtype: int64
Saved to: output/final_training_data_v5.csv


In [33]:
df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig", low_memory=False)

print("Loaded:", len(df), "rows")

# 1. Missing Position count
missing_pos = df["Position"].isna().sum()
print(f"Missing Position values: {missing_pos}")

# 2. Sample rows with missing Position
print("Sample rows with missing Position:")
display(df[df["Position"].isna()].head(20))

# 3. Frequency per season
# Using Player UUID as the count column since Player Name is missing
print("Missing positions per season:")
if "season" in df.columns:
    display(df[df["Position"].isna()]
            .groupby("season")["Player UUID"]
            .count()
            .sort_index())

# 4. Frequency per Team ID
# Using Player Team ID as the grouping column
print("Missing positions per Player Team ID:")
if "Player Team ID" in df.columns:
    display(df[df["Position"].isna()]
            .groupby("Player Team ID")["Player UUID"]
            .count()
            .sort_values(ascending=False)
            .head(20))

# 5. element_type breakdown
if "element_type" in df.columns:
    print("element_type breakdown for missing Position:")
    display(
        df[df["Position"].isna()]
        .groupby("element_type")["Player UUID"]
        .count()
    )
else:
    print("No element_type column found in the dataset.")

Loaded: 179756 rows
Missing Position values: 312
Sample rows with missing Position:


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Web Name Norm,Code,Player Team ID,Team_Total_Points_GW,Team_Total_Points_CUM,Team_Total_Points,Player_Season_Points,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct,Team_Contribution_Rank_GW
8901,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,23,0,0,0,0,0,0,0.0,...,nuno,747.0,16,14,1069,1055.0,0.0,0.00,0.00,0.3429
8902,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,24,0,0,0,0,0,15,0.0,...,nuno,747.0,16,123,1192,1069.0,0.0,12.20,0.00,0.0857
8903,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,25,0,0,0,0,0,1,0.0,...,nuno,747.0,16,32,1224,1192.0,15.0,3.12,1.26,0.2000
8904,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,26,0,0,0,0,0,3,0.0,...,nuno,747.0,16,36,1260,1224.0,16.0,8.33,1.31,0.1714
8905,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,27,0,0,0,0,0,5,0.0,...,nuno,747.0,16,57,1317,1260.0,19.0,8.77,1.51,0.1429
8906,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,28,0,0,0,0,0,9,0.0,...,nuno,747.0,16,72,1389,1317.0,24.0,12.50,1.82,0.0571
8907,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,29,0,0,0,0,0,10,0.0,...,nuno,747.0,16,67,1456,1389.0,33.0,14.93,2.38,0.0857
8908,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,30,0,0,0,0,0,9,0.0,...,nuno,747.0,16,69,1525,1456.0,43.0,13.04,2.95,0.0571
8909,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,31,0,0,0,0,0,1,0.0,...,nuno,747.0,16,28,1553,1525.0,52.0,3.57,3.41,0.2571
8910,0b79905d-e9eb-48ff-9a26-ade65bd4aeb3,2024-25,32,0,0,0,0,0,0,0.0,...,nuno,747.0,16,25,1578,1553.0,53.0,0.00,3.41,0.4000


Missing positions per season:


season
2024-25    312
Name: Player UUID, dtype: int64

Missing positions per Player Team ID:


Player Team ID
4     16
3     16
6     16
5     16
19    16
20    16
8     16
9     16
11    16
10    16
16    16
14    16
18    16
17    16
15    15
1     15
13    15
12    15
2     14
7     14
Name: Player UUID, dtype: int64

No element_type column found in the dataset.


In [34]:
import pandas as pd
from pathlib import Path

print("📂 Loading training_data_v4_with_contrib.csv ...")
df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig")
print("   → Rows:", len(df))

# ------------------------------------------------------------
# 1️⃣ Load players_raw for ALL seasons
# ------------------------------------------------------------

DATA_ROOT = Path("data")

players_raw_list = []

for season_folder in sorted(DATA_ROOT.iterdir()):
    pr_path = season_folder / "players_raw.csv"
    if pr_path.exists():
        print(f"📥 Loading → {pr_path}")
        temp = pd.read_csv(pr_path)

        # normalize ID column
        if "element" in temp.columns:
            temp = temp.rename(columns={"element": "Code"})
        elif "id" in temp.columns:
            temp = temp.rename(columns={"id": "Code"})
        else:
            continue

        if "element_type" not in temp.columns:
            print("⚠ Missing element_type → skipping")
            continue

        temp = temp[["Code", "element_type"]]
        temp["Code"] = pd.to_numeric(temp["Code"], errors="coerce")
        players_raw_list.append(temp)

players_raw = pd.concat(players_raw_list, ignore_index=True).drop_duplicates("Code")

print("✔ Total players_raw combined:", len(players_raw))

# ------------------------------------------------------------
# 2️⃣ Merge positions back into training dataset
# ------------------------------------------------------------

df["Code"] = pd.to_numeric(df["Code"], errors="coerce")

df_fix = df.merge(players_raw, on="Code", how="left")

POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
df_fix["Position"] = df_fix["element_type"].map(POS_MAP)

missing_after = df_fix["Position"].isna().sum()
print("\n🔍 Missing positions AFTER FIX:", missing_after)

# ------------------------------------------------------------
# 3️⃣ Save final fixed version
# ------------------------------------------------------------
out_path = "output/training_data_v6_positions_fixed.csv"
df_fix.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n💾 Saved → {out_path}")


📂 Loading training_data_v4_with_contrib.csv ...


   → Rows: 179756
📥 Loading → data\2018-19\players_raw.csv
📥 Loading → data\2019-20\players_raw.csv
📥 Loading → data\2020-21\players_raw.csv
📥 Loading → data\2021-22\players_raw.csv
📥 Loading → data\2022-23\players_raw.csv
📥 Loading → data\2023-24\players_raw.csv
📥 Loading → data\2024-25\players_raw.csv
📥 Loading → data\2025-26\players_raw.csv
✔ Total players_raw combined: 866

🔍 Missing positions AFTER FIX: 26396

💾 Saved → output/training_data_v6_positions_fixed.csv


In [35]:
# %% [Cell: Final Processing - Cleaning Ghost Players & 0-Min Rows]

# 1. Define final output path
CLEAN_OUT = OUTPUT_DIR / "dataset_no0min.csv"

print(f"Preparing final dataset from seasons: {df_final['season'].unique()}")

# --- STEP A: REMOVE GHOST PLAYERS (The 2025-26 Logic) ---
# We identify players who have 0 total minutes in the current season.
# If a player hasn't touched the pitch by GW29, they shouldn't be in our forecast.

current_season_mask = df_final["season"] == "2025-26"
active_this_season = df_final[current_season_mask].groupby("Player UUID")["Minutes Played"].sum()
active_uuids = active_this_season[active_this_season > 0].index

# Keep historical data, but for the current season, only keep players with > 0 total mins
df_filtered = df_final[
    (~current_season_mask) | (df_final["Player UUID"].isin(active_uuids))
].copy()

# --- STEP B: FILTER INACTIVE ROWS (Minutes > 0) ---
# Filter to keep only rows where players actually played in those specific weeks
df_dataset_no0min = df_filtered[df_filtered["Minutes Played"] > 0].copy()

print(f"Original records: {len(df_final):,}")
print(f"Records after Ghost Removal: {len(df_filtered):,}")
print(f"Final records (Minutes > 0): {len(df_dataset_no0min):,}")

# Save to output folder
df_dataset_no0min.to_csv(CLEAN_OUT, index=False, encoding="utf-8-sig")

print(f"SUCCESS!")
print(f"File saved to: {CLEAN_OUT}")

Preparing final dataset from seasons: ['2024-25' '2025-26' '2020-21' '2021-22' '2023-24' '2022-23']
Original records: 174,307
Records after Ghost Removal: 164,774
Final records (Minutes > 0): 70,866
SUCCESS!
File saved to: output\dataset_no0min.csv
